# 07. Denmark cabbage test supplement

Collect and audit 2023 Danish parcels declared as white or red cabbage. Select the project Python kernel and click **Restart Kernel → Run All**. Rerun with the same settings if Earth Engine jobs are still pending.

[Denmark Marker dataset](https://source.coop/fiboa/denmark) · [2023 crop codes](https://www.retsinformation.dk/eli/lta/2023/73)

## 1. Setup and configuration

Choose a Python 3.12+ kernel, review the settings below, then **Restart Kernel → Run All**. Opening the notebook starts no work. Rerun with the same settings to resume saved batches and jobs. Set `run_satellite_collection = False` for preparation without satellite calls; preparation can still save selections.


In [ ]:
# Review these settings before Run All.
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup
from crop_pipeline.preprocessing_workflow import paths
from crop_pipeline.sources import adaptive, annual, denmark, supplemental

run_satellite_collection = True
submit_only = False
parcels_per_batch = 50
extra_batches = 1  # Initial batch plus one extra slot.
settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
_, report, _ = paths(root)
archive = annual.archive_root(settings, report, repository_root=root)

## 2. Obtain references

The first run downloads about 400 MB and caches the filtered cabbage polygons locally. Later runs use the cached snapshot.

In [ ]:
references = root / "data/external/denmark_marker/cabbage_2023.geojson"
snapshot = denmark.load_cabbage(references)

## 3. Validate references

Count valid, distinct polygons by normalized geometry. The source can reuse field IDs.

In [ ]:
features = snapshot["features"]
print("Declared cabbage records:", snapshot["matching_records"])
print("Unique valid cabbage polygons:", len(features))

## 4. Prepare the collection round

Screen candidates against existing development and reserved CropHarvest test footprints. Resume saved plans and select unused parcels within the existing batch limits. Preserve the separate supplemental test campaign and locked partitions.


In [ ]:
batches = supplemental.prepare(
    "denmark_marker", report, archive, features, references,
    limit=parcels_per_batch, extra_batches=extra_batches,
)

## 5. Collect annual imagery

The default submits exports and retrieves available results. Keep the laptop on until Run All returns. If jobs remain pending, rerun with the same settings to resume them.

In [ ]:
project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


## 6. Check usable green-up parcels

The audit uses completed local batches and the existing quality and overlap checks. Rerun after downloads finish to get the count for the full Denmark selection.

In [ ]:
print("Collection status:", result["status"])
if run_satellite_collection and not submit_only:
    import csv
    from crop_pipeline import cabbage_test_check

    check_folder = cabbage_test_check.run(root)
    with (check_folder / "window_counts.csv").open() as handle:
        counts = next(row for row in csv.DictReader(handle)
                      if row["test_cohort"] == "denmark_marker_cabbage_2023_v1"
                      and row["window"] == "aligned" and row["population"] == "full")
    print("Denmark usable green-up parcels:", counts["usable_parcels"])
    print("Denmark usable green-up pixels:", counts["usable_pixels"])
    print("Audit:", check_folder)

## Organized outputs

Refresh the linked output folders without moving downloads or changing saved jobs. Run **08_dataset_summary.ipynb** to record saved batches and selections. Supplemental test campaigns remain separate from primary JECAM.


In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")

## Evidence and interpretation

Counts describe selected references and collection progress. Imagery and window checks determine usable samples. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.
